# GECKO v2 Whole-Genome Paper Benchmark — Legacy GECKO v1 (`master`) vs. C `gecko_fast` vs. Rust v2 (`cpu` / `wgpu` / `cuda`)

End-to-end reproducible benchmark across all **24 homologous Human (`GRCh38.p14`) vs. Chimpanzee (`Clint_PTRv2`) chromosome pairs** (`chr1`–`chr22`, `chrX`, `chrY`, pairing Human `chr2` against Chimpanzee `chr2A + chr2B`), plus optional memory-bounded slab-streaming and `(WL, FL)` sensitivity ablations for publication:

1. **`legacy_v1` (Original GECKO v1, `master` branch — Torreno & Trelles, *BMC Bioinformatics* 2015)**: 19-binary disk-based pipeline (`reverseComplement` + 3× `words`/`sortWords`/`w2hd` + 2× `hits`/`sortHits`/`filterHits`/`FragHits` + `combineFrags`), measuring wall time, sub-phase breakdown, peak RSS, and intermediate temp-disk footprint.
2. **`c_fast` (C `gecko_fast`, `gpu-backends` branch)**: Multi-threaded C (`-O3 -march=native`) in-memory canonical Hash-CSR + 64-wide prefetched streaming query + 2-tier diagonal cache + 1-byte/bp ASCII AVX2 extension.
3. **`cpu` (GECKO v2 Rust CPU, `rust-rewrite` branch)**: Portable unified Rust binary with 2-bit/bp `PackedSeq`, AVX2/SSE2/BMI2 FASTA packing, parallel page-faulting Hash-CSR/Direct-CSR, chunk-dispatched `avx2+bmi2+popcnt` inline probing & 2-tier diagonal cache, 2-bit AVX2/BMI2 prefix-sum extension, and zero-spill Reference-Slab Streaming.
4. **`wgpu` (GECKO v2 Rust WGPU, `rust-rewrite` branch)**: Cross-vendor zero-copy 2-bit Vulkan/Metal/DX12 WGSL compute shader backend (where adapter limits permit).
5. **`cuda` (GECKO v2 Rust + NVIDIA CUDA/CUB, `rust-rewrite` branch)**: Native GPU zero-copy 2-bit `PackedSeq`, word-level GPU revcomp, CUB radix Hash-CSR index, 1-bit `--freq-limit` dropped-position bitmap + lock-free maximal seed filter, adaptive sub-tiling, and 32-bp warp-broadcast extension.

**Before running:** Select `Runtime -> Change runtime type -> T4 GPU` (or `L4` / `A100`).

In [ ]:
# 1. Verify GPU, CPU, RAM, disk, CUDA/Vulkan environment, and Rust toolchain
!nvidia-smi || true
!nvcc --version || true
!which vulkaninfo && vulkaninfo --summary | head -n 30 || true
!which cargo || (curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y)
import os
os.environ['PATH'] = os.path.expanduser('~/.cargo/bin') + ':' + os.environ['PATH']
!rustc --version && gcc --version | head -n 1 && lscpu | grep -E "Model name|CPU\(s\):" | head -n 3 && free -h && df -h /content

In [ ]:
# 2. Clone & build all three baselines: GECKO v2 Rust (`rust-rewrite`), C `gecko_fast` (`gpu-backends`), and Legacy GECKO v1 (`master`)
import os, subprocess

REPO_URL = 'https://github.com/otorreno/gecko.git'
RUST_BRANCH = 'rust-rewrite'

!rm -rf /content/gecko /content/gecko_c /content/gecko_v1
!git clone --depth 1 -b {RUST_BRANCH} {REPO_URL} /content/gecko
%cd /content/gecko

# Build portable Rust v2 binary (runtime AVX2/BMI2/POPCNT dispatch + CUDA when nvcc is available)
env = os.environ.copy()
has_nvcc = subprocess.run(['which', 'nvcc'], capture_output=True).returncode == 0
if has_nvcc:
    env['GECKO_CUDA_ARCH'] = '-gencode arch=compute_75,code=sm_75 -gencode arch=compute_80,code=sm_80 -gencode arch=compute_89,code=sm_89'
    build_cmd = ['cargo', 'build', '--release', '--bin', 'gecko', '--features', 'cuda']
else:
    build_cmd = ['cargo', 'build', '--release', '--bin', 'gecko']

print('Running:', ' '.join(build_cmd))
subprocess.run(build_cmd, env=env, check=True)
!./target/release/gecko --version
!./target/release/gecko backends

# Build C `gecko_fast` from `gpu-backends` (-O3 -march=native)
!git clone --depth 1 -b gpu-backends {REPO_URL} /content/gecko_c
subprocess.run([
    'gcc', '-O3', '-D_FILE_OFFSET_BITS=64', '-march=native', '-pthread',
    '/content/gecko_c/src/gecko_fast.c',
    '/content/gecko_c/src/comparisonFunctions.c',
    '/content/gecko_c/src/commonFunctions.c',
    '-lm', '-o', '/content/gecko_c/gecko_fast',
], check=True)
print('Built C gecko_fast -> /content/gecko_c/gecko_fast')

# Build original published Legacy GECKO v1 from `master`
!git clone --depth 1 -b master {REPO_URL} /content/gecko_v1
subprocess.run([
    'make', '-C', '/content/gecko_v1/src',
    'reverseComplement', 'words', 'sortWords', 'w2hd',
    'hits', 'sortHits', 'filterHits', 'FragHits', 'combineFrags',
], check=True)
print('Built Legacy GECKO v1 binaries -> /content/gecko_v1/bin/')

In [ ]:
# 3. Configure whole-genome Human vs. Chimpanzee benchmark and active backends
import os, subprocess

GECKO_BIN = '/content/gecko/target/release/gecko'
C_FAST_BIN = '/content/gecko_c/gecko_fast'
LEGACY_V1_BIN = '/content/gecko_v1/bin'
WORK_DIR = '/content/wg_bench'
RESULTS_JSON = '/content/wg_benchmark_results.json'
os.makedirs(WORK_DIR, exist_ok=True)

# Comparison parameters (identical across all backends)
WORD_LENGTH = 32
MIN_LENGTH = 1000
MIN_SIMILARITY = 75
FREQ_LIMIT = 20
MEMORY_BUDGET = '4G'  # Fits all 24 chromosomes 100% in RAM with 2-bit PackedSeq (0 disk spill)

# Full 24 homologous chromosome pairs (chr2 pairs Human chr2 against Chimp chr2A + chr2B).
CHROMOSOMES = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']

# Toggle baselines:
INCLUDE_LEGACY_V1 = True
INCLUDE_C_FAST = True

# Legacy GECKO v1 is disk-based (~3.5 min on chr21, ~4-5 h across all 24 chrs on 2 vCPUs).
# - Set LEGACY_V1_CHROMOSOMES = None to run Legacy v1 on ALL 24 chromosomes (safe with RESUME = True).
# - Or pick a representative small/medium/large subset (~45 min) for fast turnaround, e.g.:
#   LEGACY_V1_CHROMOSOMES = ['chr21', 'chr22', 'chrY', 'chr19', 'chr13', 'chr7', 'chr1']
LEGACY_V1_CHROMOSOMES = None

# Keep RESUME = True so progress is checkpointed after EVERY backend & chromosome!
RESUME = True
# Set CHECK_PARITY = True to verify every backend's exact & overlap recall against Rust CPU
CHECK_PARITY = True

probe_out = subprocess.run([GECKO_BIN, 'backends'], capture_output=True, text=True, check=True).stdout
print(probe_out)

rust_backends = []
for b in ['cpu', 'wgpu', 'cuda']:
    for line in probe_out.splitlines():
        if line.strip().startswith(f'{b} ') and 'AVAILABLE' in line:
            rust_backends.append(b)
            break

BACKENDS = []
if INCLUDE_LEGACY_V1 and os.path.exists(os.path.join(LEGACY_V1_BIN, 'FragHits')):
    BACKENDS.append('legacy_v1')
if INCLUDE_C_FAST and os.path.exists(C_FAST_BIN):
    BACKENDS.append('c_fast')
BACKENDS.extend(rust_backends)

legacy_set = set(CHROMOSOMES if LEGACY_V1_CHROMOSOMES is None else LEGACY_V1_CHROMOSOMES)
print('Selected backends:', BACKENDS)
print(f'Chromosomes ({len(CHROMOSOMES)}):', ', '.join(CHROMOSOMES))
if 'legacy_v1' in BACKENDS:
    print(f'Legacy GECKO v1 enabled on {len(legacy_set)}/{len(CHROMOSOMES)} chromosomes:',
          'ALL' if len(legacy_set) == len(CHROMOSOMES) else ', '.join(sorted(legacy_set)))

In [ ]:
# 4. Run full 24-chromosome Human (GRCh38.p14) vs. Chimpanzee (Clint_PTRv2) benchmark across all backends
import gzip, json, os, re, resource, shutil, struct, subprocess, tempfile, time
from concurrent.futures import ThreadPoolExecutor

HS_BASE = (
    'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/000/001/405/'
    'GCF_000001405.40_GRCh38.p14/GCF_000001405.40_GRCh38.p14_assembly_structure/'
    'Primary_Assembly/assembled_chromosomes/FASTA'
)
PT_BASE = (
    'https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/002/880/755/'
    'GCF_002880755.1_Clint_PTRv2/GCF_002880755.1_Clint_PTRv2_assembly_structure/'
    'Primary_Assembly/assembled_chromosomes/FASTA'
)

def curl_download(url, dst):
    subprocess.run([
        'curl', '-fsSL',
        '--connect-timeout', '15',
        '--speed-time', '20',
        '--speed-limit', '50000',
        '--retry', '5',
        '--retry-delay', '2',
        '--retry-all-errors',
        '-o', dst, url,
    ], check=True)

def gunzip_to(src_gz, dst_fa, mode='wb'):
    with gzip.open(src_gz, 'rb') as fin, open(dst_fa, mode) as fout:
        shutil.copyfileobj(fin, fout, 1 << 20)

def download_pair(ch):
    hs_fa = os.path.join(WORK_DIR, f'hs_{ch}.fna')
    pt_fa = os.path.join(WORK_DIR, f'pt_{ch}.fna')
    hs_gz = hs_fa + '.gz'
    curl_download(f'{HS_BASE}/{ch}.fna.gz', hs_gz)
    gunzip_to(hs_gz, hs_fa, 'wb')
    os.remove(hs_gz)

    if ch == 'chr2':
        for part, mode in [('chr2A', 'wb'), ('chr2B', 'ab')]:
            p_gz = os.path.join(WORK_DIR, f'pt_{part}.fna.gz')
            curl_download(f'{PT_BASE}/{part}.fna.gz', p_gz)
            gunzip_to(p_gz, pt_fa, mode)
            os.remove(p_gz)
    else:
        pt_gz = pt_fa + '.gz'
        curl_download(f'{PT_BASE}/{ch}.fna.gz', pt_gz)
        gunzip_to(pt_gz, pt_fa, 'wb')
        os.remove(pt_gz)
    return hs_fa, pt_fa

def parse_frags(path):
    frags = []
    with open(path, 'rb') as f:
        n0, n1 = struct.unpack('>QQ', f.read(16))
        while len(b := f.read(93)) == 93:
            diag, xs, ys, xe, ye, length, ident, score, sim, sx, sy, blk, st = struct.unpack('>qQQQQQQQfQQqB', b)
            ch = chr(st)
            if ch == 'r':
                ys = n1 - ys - 1
                ye = n1 - ye - 1
            frags.append((xs, ys, xe, ye, ch, length, score, ident, sx, sy))
    return frags

def compute_parity(ref_frags, test_frags):
    sr, st = set(ref_frags), set(test_frags)
    if not sr:
        return 100.0, 100.0
    exact = len(sr & st)
    by_diag = {}
    for fr in test_frags:
        dk = (fr[4], fr[0] - fr[1] if fr[4] == 'f' else fr[0] + fr[1], fr[8], fr[9])
        by_diag.setdefault(dk, []).append((fr[0], fr[2]))
    cov = 0
    for fr in (sr - st):
        dk = (fr[4], fr[0] - fr[1] if fr[4] == 'f' else fr[0] + fr[1], fr[8], fr[9])
        for a, b in by_diag.get(dk, ()):
            if min(fr[2], b) - max(fr[0], a) + 1 >= 0.9 * (fr[2] - fr[0] + 1):
                cov += 1
                break
    return 100.0 * exact / len(sr), 100.0 * (exact + cov) / len(sr)

def _raise_stack_for_v1():
    # Legacy reverseComplement.c allocates `long offset[1_000_000]` (8 MiB) on the stack.
    soft, hard = resource.getrlimit(resource.RLIMIT_STACK)
    want = 64 * 1024 * 1024
    if soft != resource.RLIM_INFINITY and soft < want:
        target = want if hard == resource.RLIM_INFINITY else min(want, hard)
        resource.setrlimit(resource.RLIMIT_STACK, (target, hard))

def run_legacy_v1(ch, hs_fa, pt_fa):
    outdir = os.path.join(WORK_DIR, f'out_{ch}_legacy_v1')
    shutil.rmtree(outdir, ignore_errors=True)
    os.makedirs(outdir, exist_ok=True)
    n_workers = min(2, os.cpu_count() or 2)
    pt_rc = os.path.join(outdir, 'Yrc.fna')

    def step(cmd):
        with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
            p = subprocess.Popen(cmd, cwd=outdir, preexec_fn=_raise_stack_for_v1, stdout=out_f, stderr=err_f)
            _, status, ru = os.wait4(p.pid, 0)
            if status != 0:
                out_f.seek(0); err_f.seek(0)
                msg = (err_f.read() or out_f.read()).decode('utf-8', errors='replace').strip().splitlines()
                raise RuntimeError(f"{os.path.basename(cmd[0])} failed (status={status}): {msg[-1] if msg else ''}")
            return ru.ru_maxrss / 1024.0

    def fsize_mib(name):
        p = os.path.join(outdir, name)
        return os.path.getsize(p) / (1024 * 1024) if os.path.exists(p) else 0.0

    def dir_mib():
        tot = 0
        for root, _, files in os.walk(outdir):
            for fn in files:
                fp = os.path.join(root, fn)
                if os.path.exists(fp):
                    tot += os.path.getsize(fp)
        return tot / (1024 * 1024)

    t0 = time.perf_counter()
    peak_rss = 0.0
    peak_disk = 0.0
    total_written = 0.0

    try:
        # Phase 1: reverseComplement
        t_rc0 = time.perf_counter()
        peak_rss = max(peak_rss, step([os.path.join(LEGACY_V1_BIN, 'reverseComplement'), pt_fa, pt_rc]))
        revcomp_s = time.perf_counter() - t_rc0
        total_written += fsize_mib('Yrc.fna')

        # Phase 2: 3x disk-based dictionaries (X, Y, Yrc)
        def build_dict(fa_path, pref):
            r1 = step([os.path.join(LEGACY_V1_BIN, 'words'), fa_path, f'{pref}.words.unsort'])
            w_unsort = fsize_mib(f'{pref}.words.unsort')
            r2 = step([os.path.join(LEGACY_V1_BIN, 'sortWords'), '10000000', '32', f'{pref}.words.unsort', f'{pref}.words.sort'])
            w_sort = fsize_mib(f'{pref}.words.sort')
            r3 = step([os.path.join(LEGACY_V1_BIN, 'w2hd'), f'{pref}.words.sort', pref])
            w_hd = fsize_mib(f'{pref}.d2hW') + fsize_mib(f'{pref}.d2hP')
            p_disk = dir_mib()
            # Clean consumed .words.sort immediately so Colab disk never fills on chr1/chr2
            for stale in (f'{pref}.words.unsort', f'{pref}.words.sort'):
                sp = os.path.join(outdir, stale)
                if os.path.exists(sp):
                    os.remove(sp)
            return max(r1, r2, r3), p_disk, (w_unsort + w_sort + w_hd)

        t_d0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            futs = [
                ex.submit(build_dict, hs_fa, 'X'),
                ex.submit(build_dict, pt_fa, 'Y'),
                ex.submit(build_dict, pt_rc, 'Yrc'),
            ]
            for fut in futs:
                r_m, d_m, w_m = fut.result()
                peak_rss = max(peak_rss, r_m)
                peak_disk = max(peak_disk, d_m)
                total_written += w_m
        dict_s = time.perf_counter() - t_d0

        # Phase 3: 2x hits -> sortHits -> filterHits -> FragHits (forward & reverse)
        def run_strand(y_fa, y_pref, strand):
            base = f'X-{y_pref}-K{WORD_LENGTH}'
            r1 = step([os.path.join(LEGACY_V1_BIN, 'hits'), 'X', y_pref, f'{base}.hits', str(FREQ_LIMIT), str(WORD_LENGTH)])
            w1 = fsize_mib(f'{base}.hits')
            r2 = step([os.path.join(LEGACY_V1_BIN, 'sortHits'), '10000000', '32', f'{base}.hits', f'{base}.hits.sorted'])
            w2 = fsize_mib(f'{base}.hits.sorted')
            r3 = step([os.path.join(LEGACY_V1_BIN, 'filterHits'), f'{base}.hits.sorted', f'{base}.hits.sorted.filtered', str(WORD_LENGTH)])
            w3 = fsize_mib(f'{base}.hits.sorted.filtered')
            p_disk = dir_mib()
            for stale in (f'{base}.hits', f'{base}.hits.sorted'):
                sp = os.path.join(outdir, stale)
                if os.path.exists(sp):
                    os.remove(sp)
            r4 = step([
                os.path.join(LEGACY_V1_BIN, 'FragHits'), hs_fa, y_fa,
                f'{base}.hits.sorted.filtered', f'{ch}-s{strand}.frags',
                str(MIN_LENGTH), str(MIN_SIMILARITY), str(WORD_LENGTH), '1', strand,
            ])
            return max(r1, r2, r3, r4), p_disk, (w1 + w2 + w3)

        t_c0 = time.perf_counter()
        with ThreadPoolExecutor(max_workers=n_workers) as ex:
            f_fwd = ex.submit(run_strand, pt_fa, 'Y', 'f')
            f_rev = ex.submit(run_strand, pt_rc, 'Yrc', 'r')
            for fut in (f_fwd, f_rev):
                r_m, d_m, w_m = fut.result()
                peak_rss = max(peak_rss, r_m)
                peak_disk = max(peak_disk, d_m)
                total_written += w_m
        cmp_s = time.perf_counter() - t_c0

        # Phase 4: combineFrags
        t_m0 = time.perf_counter()
        final_frags = os.path.join(outdir, f'{ch}.frags')
        peak_rss = max(peak_rss, step([
            os.path.join(LEGACY_V1_BIN, 'combineFrags'),
            f'{ch}-sf.frags', f'{ch}-sr.frags', f'{ch}.frags',
        ]))
        merge_s = time.perf_counter() - t_m0
        wall_s = time.perf_counter() - t0

        frags = parse_frags(final_frags)
        n_fwd = sum(1 for fr in frags if fr[4] == 'f')
        n_rev = len(frags) - n_fwd
        with open(final_frags, 'rb') as fp:
            q_bases, r_bases = struct.unpack('>QQ', fp.read(16))

        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend_label': f'legacy_v1 (C 2015 disk-based, {n_workers} workers)',
            'wall_s': round(wall_s, 3),
            'load_s': 0.0,
            'compute_s': round(wall_s, 3),
            'write_s': round(merge_s, 3),
            'revcomp_s': round(revcomp_s, 3),
            'dict_s': round(dict_s, 3),
            'cmp_s': round(cmp_s, 3),
            'rss_mib': round(peak_rss, 1),
            'temp_disk_mib': round(peak_disk, 1),
            'total_written_mib': round(total_written, 1),
            'fragments': len(frags),
            'forward_fragments': n_fwd,
            'reverse_fragments': n_rev,
            'spilled_runs': 1,
            'query_bases': q_bases,
            'reference_bases': r_bases,
        }, (frags if CHECK_PARITY else None)
    except Exception as exc:
        shutil.rmtree(outdir, ignore_errors=True)
        return {'ok': False, 'error': str(exc)}, None

def run_backend(backend, ch, hs_fa, pt_fa):
    if backend == 'legacy_v1':
        return run_legacy_v1(ch, hs_fa, pt_fa)

    outdir = os.path.join(WORK_DIR, f'out_{ch}_{backend}')
    os.makedirs(outdir, exist_ok=True)
    if backend == 'c_fast':
        prefix_path = os.path.join(outdir, ch)
        threads = str(os.cpu_count() or 2)
        cmd = [
            C_FAST_BIN, hs_fa, pt_fa, prefix_path,
            str(MIN_LENGTH), str(MIN_SIMILARITY), str(WORD_LENGTH), '1',
            threads, str(FREQ_LIMIT),
        ]
    else:
        cmd = [
            GECKO_BIN, 'compare', hs_fa, pt_fa,
            '-o', outdir,
            '--prefix', ch,
            '--word-length', str(WORD_LENGTH),
            '-l', str(MIN_LENGTH),
            '-s', str(MIN_SIMILARITY),
            '--freq-limit', str(FREQ_LIMIT),
            '-m', MEMORY_BUDGET,
            '--backend', backend,
            '--format', 'frags',
            '--json-stats',
        ]
    with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
        t0 = time.perf_counter()
        p = subprocess.Popen(cmd, stdout=out_f, stderr=err_f)
        _, status, rusage = os.wait4(p.pid, 0)
        wall_s = time.perf_counter() - t0
        out_f.seek(0)
        err_f.seek(0)
        stdout = out_f.read().decode('utf-8', errors='replace')
        stderr = err_f.read().decode('utf-8', errors='replace')
        rss_mib = rusage.ru_maxrss / 1024.0

    if status != 0:
        shutil.rmtree(outdir, ignore_errors=True)
        err_lines = (stderr.strip() or stdout.strip()).splitlines()
        err_msg = err_lines[-1] if err_lines else f'exit {status}'
        return {'ok': False, 'error': err_msg}, None

    frags_path = os.path.join(outdir, f'{ch}.frags')
    frags = parse_frags(frags_path) if CHECK_PARITY else None

    if backend == 'c_fast':
        combined = stdout + '\n' + stderr
        m_seq = re.search(r'SeqX:\s*(\d+)\s*bp.*SeqY:\s*(\d+)\s*bp', combined)
        m_frag = re.search(r'Fragments:\s*(\d+)\s*\(F:(\d+)\s*R:(\d+)\)', combined)
        m_time = re.search(
            r'Timing:\s*load=([0-9.]+)s\s+index=([0-9.]+)s\s+query\+ext=([0-9.]+)s\s+write=([0-9.]+)s',
            combined,
        )
        load_s = float(m_time.group(1)) if m_time else 0.0
        compute_s = (float(m_time.group(2)) + float(m_time.group(3))) if m_time else wall_s
        write_s = float(m_time.group(4)) if m_time else 0.0
        n_frags = int(m_frag.group(1)) if m_frag else (len(frags) if frags is not None else 0)
        n_fwd = int(m_frag.group(2)) if m_frag else 0
        n_rev = int(m_frag.group(3)) if m_frag else 0
        q_bases = int(m_seq.group(1)) if m_seq else 0
        r_bases = int(m_seq.group(2)) if m_seq else 0
        shutil.rmtree(outdir, ignore_errors=True)
        return {
            'ok': True,
            'backend_label': f'c_fast (C AVX2 -march=native, {os.cpu_count() or 2} threads)',
            'wall_s': round(wall_s, 3),
            'load_s': round(load_s, 3),
            'compute_s': round(compute_s, 3),
            'write_s': round(write_s, 3),
            'rss_mib': round(rss_mib, 1),
            'temp_disk_mib': 0.0,
            'total_written_mib': 0.0,
            'fragments': n_frags,
            'forward_fragments': n_fwd,
            'reverse_fragments': n_rev,
            'spilled_runs': 0,
            'query_bases': q_bases,
            'reference_bases': r_bases,
        }, frags

    st = json.loads(stdout)
    shutil.rmtree(outdir, ignore_errors=True)
    return {
        'ok': True,
        'backend_label': st.get('backend', backend),
        'wall_s': round(wall_s, 3),
        'load_s': round(st.get('load_secs', 0.0), 3),
        'compute_s': round(st.get('compute_secs', wall_s), 3),
        'write_s': round(st.get('write_secs', 0.0), 3),
        'rss_mib': round(rss_mib, 1),
        'temp_disk_mib': 0.0,
        'total_written_mib': 0.0,
        'fragments': st['fragments'],
        'forward_fragments': st['forward_fragments'],
        'reverse_fragments': st['reverse_fragments'],
        'spilled_runs': st['spilled_runs'],
        'query_bases': st['query_bases'],
        'reference_bases': st['reference_bases'],
    }, frags

# Load existing results when resuming
results = {}
if RESUME and os.path.exists(RESULTS_JSON):
    with open(RESULTS_JSON) as f:
        results = json.load(f)
    print(f'Resuming with {len(results)} chromosomes already recorded.')

for idx, ch in enumerate(CHROMOSOMES, 1):
    existing = results.get(ch, {})
    active_for_ch = [b for b in BACKENDS if b != 'legacy_v1' or ch in legacy_set]
    needed = [b for b in active_for_ch if b not in existing]
    if not needed:
        print(f'[{idx:2d}/{len(CHROMOSOMES)}] {ch}: already complete, skipping.')
        continue

    print(f'\n[{idx:2d}/{len(CHROMOSOMES)}] {ch} — downloading FASTAs...', flush=True)
    hs_fa, pt_fa = download_pair(ch)

    # Always run Rust `cpu` first when checking parity so every other backend is scored against it
    run_list = list(needed)
    if 'cpu' in run_list:
        run_list.remove('cpu')
        run_list = ['cpu'] + run_list
    elif CHECK_PARITY and any(b != 'cpu' for b in run_list):
        run_list = ['cpu'] + run_list

    cpu_frags = None
    row = dict(existing)
    for b in run_list:
        rec, frags = run_backend(b, ch, hs_fa, pt_fa)
        if b == 'cpu' and rec['ok']:
            cpu_frags = frags
        elif CHECK_PARITY and rec['ok'] and cpu_frags is not None and frags is not None:
            ex_pct, rec_pct = compute_parity(cpu_frags, frags)
            rec['exact_pct'] = round(ex_pct, 3)
            rec['recall_pct'] = round(rec_pct, 4)
            if b == 'legacy_v1':
                # Also record what % of legacy_v1's fragments Rust CPU recovers!
                v1_ex, v1_rec = compute_parity(frags, cpu_frags)
                rec['cpu_recovers_v1_pct'] = round(v1_rec, 4)

        if b in needed:
            row[b] = rec

        if rec['ok']:
            parity_str = f" | recall={rec['recall_pct']:.3f}%" if 'recall_pct' in rec else ''
            if 'cpu_recovers_v1_pct' in rec:
                parity_str += f" (v2 recovers v1={rec['cpu_recovers_v1_pct']:.3f}%)"
            disk_str = f" | tmp={rec['temp_disk_mib']:,.0f} MiB (tot_wr={rec['total_written_mib']:,.0f} MiB)" if rec.get('temp_disk_mib', 0) > 0 else ''
            print(
                f"  [{b:<9}] wall={rec['wall_s']:7.2f}s "
                f"(load={rec['load_s']:.2f}s compute={rec['compute_s']:.2f}s) | "
                f"RSS={rec['rss_mib']:6.1f} MiB | frags={rec['fragments']:,}{disk_str}{parity_str}",
                flush=True,
            )
        else:
            print(f"  [{b:<9}] SKIPPED/ERROR: {rec['error']}", flush=True)

        # Checkpoint after every backend so long legacy_v1 runs never lose progress!
        results[ch] = row
        with open(RESULTS_JSON, 'w') as f:
            json.dump(results, f, indent=2)

    os.remove(hs_fa)
    os.remove(pt_fa)

print('\nAll chromosomes finished! Saved raw JSON to:', RESULTS_JSON)

In [ ]:
# 5. Compile per-chromosome & whole-genome tables (CSV, Markdown, and LaTeX)
import json
import pandas as pd
from IPython.display import display, Markdown

with open(RESULTS_JSON) as f:
    results = json.load(f)

rows = []
for ch in CHROMOSOMES:
    r = results.get(ch, {})
    any_ok = next((r[b] for b in BACKENDS if b in r and r[b].get('ok')), None)
    entry = {
        'Chr': 'chr2 (2A+2B)' if ch == 'chr2' else ch,
        'Human_Mb': round(any_ok['query_bases'] / 1e6, 2) if any_ok else None,
        'Chimp_Mb': round(any_ok['reference_bases'] / 1e6, 2) if any_ok else None,
    }
    v1_wall = r.get('legacy_v1', {}).get('wall_s') if r.get('legacy_v1', {}).get('ok') else None
    cpu_wall = r.get('cpu', {}).get('wall_s') if r.get('cpu', {}).get('ok') else None
    for b in BACKENDS:
        br = r.get(b, {})
        if br.get('ok'):
            entry[f'{b}_wall_s'] = br['wall_s']
            entry[f'{b}_compute_s'] = br['compute_s']
            entry[f'{b}_rss_MiB'] = br['rss_mib']
            if b == 'legacy_v1':
                entry['legacy_v1_disk_MiB'] = br.get('temp_disk_mib', 0.0)
                entry['legacy_v1_written_MiB'] = br.get('total_written_mib', 0.0)
            entry[f'{b}_frags'] = br['fragments']
            if b != 'legacy_v1' and v1_wall:
                entry[f'{b}_vs_v1'] = round(v1_wall / br['wall_s'], 2)
            if b != 'cpu' and cpu_wall:
                entry[f'{b}_vs_cpu'] = round(cpu_wall / br['wall_s'], 2)
            if 'recall_pct' in br:
                entry[f'{b}_recall_%'] = br['recall_pct']
        else:
            entry[f'{b}_wall_s'] = None
            entry[f'{b}_compute_s'] = None
            entry[f'{b}_rss_MiB'] = None
            if b == 'legacy_v1':
                entry['legacy_v1_disk_MiB'] = None
                entry['legacy_v1_written_MiB'] = None
            entry[f'{b}_frags'] = None
            if b != 'legacy_v1' and 'legacy_v1' in BACKENDS:
                entry[f'{b}_vs_v1'] = None
            if b != 'cpu':
                entry[f'{b}_vs_cpu'] = None
                entry[f'{b}_recall_%'] = None
    rows.append(entry)

df = pd.DataFrame(rows)

# Build TOTAL summary row (speedups strictly matched over chromosomes completed by BOTH backends)
total = {
    'Chr': 'TOTAL',
    'Human_Mb': round(df['Human_Mb'].sum(), 2),
    'Chimp_Mb': round(df['Chimp_Mb'].sum(), 2),
}
v1_mask = df['legacy_v1_wall_s'].notna() if 'legacy_v1_wall_s' in df else None
for b in BACKENDS:
    valid_mask = df[f'{b}_wall_s'].notna()
    n_ok = int(valid_mask.sum())
    b_tot = df[f'{b}_wall_s'].sum()
    total[f'{b}_wall_s'] = round(b_tot, 2) if n_ok > 0 else None
    total[f'{b}_compute_s'] = round(df[f'{b}_compute_s'].sum(), 2) if n_ok > 0 else None
    total[f'{b}_rss_MiB'] = round(df[f'{b}_rss_MiB'].max(), 1) if n_ok > 0 else None
    if b == 'legacy_v1':
        total['legacy_v1_disk_MiB'] = round(df['legacy_v1_disk_MiB'].max(), 1) if n_ok > 0 else None
        total['legacy_v1_written_MiB'] = round(df['legacy_v1_written_MiB'].sum(), 1) if n_ok > 0 else None
    total[f'{b}_frags'] = int(df[f'{b}_frags'].sum()) if n_ok > 0 else None
    if b != 'legacy_v1' and v1_mask is not None and f'{b}_vs_v1' in df:
        both = valid_mask & v1_mask
        if int(both.sum()) > 0:
            total[f'{b}_vs_v1'] = round(df.loc[both, 'legacy_v1_wall_s'].sum() / df.loc[both, f'{b}_wall_s'].sum(), 2)
    if b != 'cpu' and n_ok > 0 and 'cpu_wall_s' in df:
        both = valid_mask & df['cpu_wall_s'].notna()
        if int(both.sum()) > 0:
            total[f'{b}_vs_cpu'] = round(df.loc[both, 'cpu_wall_s'].sum() / df.loc[both, f'{b}_wall_s'].sum(), 2)
        if f'{b}_recall_%' in df:
            total[f'{b}_recall_%'] = round(df.loc[valid_mask, f'{b}_recall_%'].mean(), 4)

df_full = pd.concat([df, pd.DataFrame([total])], ignore_index=True)

csv_path = '/content/human_vs_chimp_backend_timings.csv'
md_path = '/content/human_vs_chimp_backend_timings.md'
tex_path = '/content/human_vs_chimp_summary.tex'
df_full.to_csv(csv_path, index=False)

def fmt_val(col, v):
    if pd.isna(v):
        return '—'
    if col.endswith('_s'):
        return f'{v:.2f}s'
    if col.endswith('_MiB'):
        return f'{v:,.1f}'
    if col.endswith('_frags'):
        return f'{int(v):,}'
    if col.endswith('_vs_v1') or col.endswith('_vs_cpu'):
        return f'{v:.2f}x'
    if col.endswith('_recall_%'):
        return f'{v:.3f}%'
    if col.endswith('_Mb'):
        return f'{v:.2f}'
    return str(v)

cols = list(df_full.columns)
md_lines = [
    f'## Whole Human (`GRCh38.p14`) vs. Chimpanzee (`Clint_PTRv2`) Backend Timings (`WL={WORD_LENGTH}, L={MIN_LENGTH}, S={MIN_SIMILARITY}%, FL={FREQ_LIMIT}`)',
    '',
    '| ' + ' | '.join(cols) + ' |',
    '| ' + ' | '.join([':---' if c == 'Chr' else '---:' for c in cols]) + ' |',
]
for _, r in df_full.iterrows():
    is_tot = r['Chr'] == 'TOTAL'
    vals = [f'**{fmt_val(c, r[c])}**' if is_tot else fmt_val(c, r[c]) for c in cols]
    md_lines.append('| ' + ' | '.join(vals) + ' |')

md_text = '\n'.join(md_lines)
with open(md_path, 'w') as f:
    f.write(md_text + '\n')

# Also write compact paper-ready LaTeX summary table
tex_rows = []
for b in BACKENDS:
    w_s = total.get(f'{b}_wall_s')
    c_s = total.get(f'{b}_compute_s')
    rss = total.get(f'{b}_rss_MiB')
    dsk = total.get('legacy_v1_disk_MiB', 0.0) if b == 'legacy_v1' else 0.0
    frg = total.get(f'{b}_frags')
    sp_v1 = total.get(f'{b}_vs_v1') if b != 'legacy_v1' else 1.0
    sp_cpu = total.get(f'{b}_vs_cpu') if b != 'cpu' else 1.0
    rec = total.get(f'{b}_recall_%') if b != 'cpu' else 100.0
    b_tex = b.replace('_', r'\_')
    tex_rows.append(
        f"    \\texttt{{{b_tex}}} & "
        f"{(f'{w_s:.2f}\\,s' if w_s else '---')} & "
        f"{(f'{c_s:.2f}\\,s' if c_s else '---')} & "
        f"{(f'{rss:,.1f}' if rss else '---')} & "
        f"{(f'{dsk:,.1f}' if dsk is not None else '0.0')} & "
        f"{(f'{int(frg):,}' if frg else '---')} & "
        f"{(f'{sp_v1:.2f}\\times' if sp_v1 else '---')} & "
        f"{(f'{sp_cpu:.2f}\\times' if sp_cpu else '---')} & "
        f"{(f'{rec:.3f}\\%' if rec else '---')} \\\\"
    )
tex_table = '\n'.join([
    r'\begin{table*}[t]',
    r'  \centering',
    r'  \caption{Whole-genome Human (\texttt{GRCh38.p14}) vs.\ Chimpanzee (\texttt{Clint\_PTRv2}) comparison across all 24 chromosome pairs ($WL=32, L=1000, S=75\%, FL=20$).}',
    r'  \label{tab:whole_genome}',
    r'  \begin{tabular}{lrrrrrrrr}',
    r'    \toprule',
    r'    Backend & Wall & Compute & Peak RSS (MiB) & Peak Disk (MiB) & Fragments & vs.\ v1 & vs.\ Rust CPU & Recall \\',
    r'    \midrule',
    *tex_rows,
    r'    \bottomrule',
    r'  \end{tabular}',
    r'\end{table*}',
])
with open(tex_path, 'w') as f:
    f.write(tex_table + '\n')

display(Markdown(md_text))
print(f'\nSaved CSV -> {csv_path}\nSaved MD  -> {md_path}\nSaved TeX -> {tex_path}')

In [ ]:
# 6. Publication-quality multi-panel figures (300 DPI PNG + vector PDF)
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(19, 5.2), gridspec_kw={'width_ratios': [2.6, 1.3, 1.3]})
colors = {
    'legacy_v1': '#64748b',
    'c_fast': '#ef4444',
    'cpu': '#2563eb',
    'wgpu': '#f59e0b',
    'cuda': '#10b981',
}
labels = {
    'legacy_v1': 'GECKO v1 (2015)',
    'c_fast': 'C gecko_fast',
    'cpu': 'GECKO v2 Rust CPU',
    'wgpu': 'GECKO v2 WGPU',
    'cuda': 'GECKO v2 CUDA',
}

# Panel A: Per-chromosome wall time
ax1 = axes[0]
x = np.arange(len(df))
w = 0.82 / max(len(BACKENDS), 1)
for i, b in enumerate(BACKENDS):
    vals = df[f'{b}_wall_s'].fillna(0).values
    ax1.bar(x + (i - (len(BACKENDS) - 1) / 2) * w, vals, width=w,
            label=labels.get(b, b), color=colors.get(b))
ax1.set_xticks(x)
ax1.set_xticklabels([c.replace(' (2A+2B)', '*') for c in df['Chr']], rotation=45, ha='right', fontsize=8)
if 'legacy_v1' in BACKENDS and df['legacy_v1_wall_s'].notna().any():
    ax1.set_yscale('log')
    ax1.set_ylabel('Wall-clock time (s, log scale)')
else:
    ax1.set_ylabel('Wall-clock time (s)')
ax1.set_title(f'A. Per-Chromosome Runtime (WL={WORD_LENGTH}, FL={FREQ_LIMIT})')
ax1.legend(fontsize=8)
ax1.grid(axis='y', alpha=0.3)

# Panel B: Total runtime & speedup
ax2 = axes[1]
tot_names = [labels.get(b, b).replace(' ', '\n') for b in BACKENDS]
tot_mins = [(total.get(f'{b}_wall_s') or 0) / 60.0 for b in BACKENDS]
bars = ax2.bar(tot_names, tot_mins, color=[colors.get(b) for b in BACKENDS])
if 'legacy_v1' in BACKENDS and total.get('legacy_v1_wall_s'):
    ax2.set_yscale('log')
    ax2.set_ylabel('Total time (min, log scale)')
else:
    ax2.set_ylabel('Total time (min)')
ax2.set_title('B. Total Genome Runtime')
ax2.grid(axis='y', alpha=0.3)
for bar, b, m in zip(bars, BACKENDS, tot_mins):
    sp = total.get(f'{b}_vs_v1') if 'legacy_v1' in BACKENDS and b != 'legacy_v1' else total.get(f'{b}_vs_cpu')
    sp_txt = f'\n({sp:.1f}x)' if sp else ''
    ax2.text(bar.get_x() + bar.get_width() / 2, m * 1.04, f'{m:.2f}m{sp_txt}',
             ha='center', va='bottom', fontsize=8, fontweight='bold')

# Panel C: Peak Host RSS & Peak Temp Disk
ax3 = axes[2]
rss_gb = [(total.get(f'{b}_rss_MiB') or 0) / 1024.0 for b in BACKENDS]
disk_gb = [(total.get('legacy_v1_disk_MiB') or 0) / 1024.0 if b == 'legacy_v1' else 0.0 for b in BACKENDS]
ix = np.arange(len(BACKENDS))
ax3.bar(ix - 0.18, rss_gb, width=0.35, label='Peak Host RSS (GiB)', color='#3b82f6')
ax3.bar(ix + 0.18, disk_gb, width=0.35, label='Peak Temp Disk (GiB)', color='#f97316')
ax3.set_xticks(ix)
ax3.set_xticklabels(tot_names, fontsize=8)
ax3.set_ylabel('GiB')
ax3.set_title('C. Memory & Temp Disk Footprint')
ax3.legend(fontsize=8)
ax3.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('/content/human_vs_chimp_paper_figures.png', dpi=300)
plt.savefig('/content/human_vs_chimp_paper_figures.pdf')
plt.show()
print('Saved /content/human_vs_chimp_paper_figures.{png,pdf}')

In [ ]:
# 7. (Optional Paper Ablation 1) Zero-Spill Reference-Slab Streaming on chr1 across --memory budgets
RUN_SLAB_ABLATION = False  # Set True to generate Table/Figure for memory-bounded slab streaming

if RUN_SLAB_ABLATION:
    import json, os, shutil, subprocess, tempfile, time
    import pandas as pd

    hs1, pt1 = download_pair('chr1')
    budgets = ['512M', '1G', '2G', '4G']
    slab_rows = []
    ref_frags = None

    for mem in budgets:
        outdir = os.path.join(WORK_DIR, f'ablation_slab_{mem}')
        shutil.rmtree(outdir, ignore_errors=True)
        os.makedirs(outdir, exist_ok=True)
        cmd = [
            GECKO_BIN, 'compare', hs1, pt1, '-o', outdir, '--prefix', f'chr1_{mem}',
            '-k', str(WORD_LENGTH), '-l', str(MIN_LENGTH), '-s', str(MIN_SIMILARITY),
            '--freq-limit', str(FREQ_LIMIT), '-m', mem, '--backend', 'cpu',
            '--format', 'frags', '--json-stats',
        ]
        with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
            t0 = time.perf_counter()
            p = subprocess.Popen(cmd, stdout=out_f, stderr=err_f)
            _, status, ru = os.wait4(p.pid, 0)
            wall_s = time.perf_counter() - t0
            assert status == 0
            out_f.seek(0)
            st = json.loads(out_f.read().decode())
            rss_mib = ru.ru_maxrss / 1024.0
        frags = parse_frags(os.path.join(outdir, f'chr1_{mem}.frags'))
        if mem == '4G' or ref_frags is None:
            ref_frags = frags
        ex_pct, rec_pct = compute_parity(ref_frags, frags)
        slab_rows.append({
            'memory_budget': mem, 'wall_s': round(wall_s, 2),
            'compute_s': round(st['compute_secs'], 2), 'peak_rss_MiB': round(rss_mib, 1),
            'spilled_runs': st['spilled_runs'], 'fragments': st['fragments'],
            'exact_parity_%': round(ex_pct, 4),
        })
        shutil.rmtree(outdir, ignore_errors=True)

    os.remove(hs1); os.remove(pt1)
    df_slab = pd.DataFrame(slab_rows)
    df_slab.to_csv('/content/ablation_slab_memory.csv', index=False)
    display(df_slab)

In [ ]:
# 8. (Optional Paper Ablation 2) Word-Length & --freq-limit Recall/Speedup Matrix on chr21 (CPU vs CUDA)
RUN_WL_FL_ABLATION = False  # Set True to generate (WL, FL) sensitivity table for the paper

if RUN_WL_FL_ABLATION and 'cuda' in BACKENDS:
    import json, os, shutil, subprocess, tempfile, time
    import pandas as pd

    hs21, pt21 = download_pair('chr21')
    grid = [(12, 20), (16, 20), (24, 20), (32, 20), (32, 100), (32, 1000)]
    ab_rows = []
    for wl, fl in grid:
        res_pair = {}
        frags_pair = {}
        for b in ['cpu', 'cuda']:
            outdir = os.path.join(WORK_DIR, f'ab_wl{wl}_fl{fl}_{b}')
            shutil.rmtree(outdir, ignore_errors=True)
            os.makedirs(outdir, exist_ok=True)
            cmd = [
                GECKO_BIN, 'compare', hs21, pt21, '-o', outdir, '--prefix', 'chr21',
                '-k', str(wl), '-l', str(MIN_LENGTH), '-s', str(MIN_SIMILARITY),
                '--freq-limit', str(fl), '-m', MEMORY_BUDGET, '--backend', b,
                '--format', 'frags', '--json-stats',
            ]
            with tempfile.TemporaryFile() as out_f, tempfile.TemporaryFile() as err_f:
                t0 = time.perf_counter()
                p = subprocess.Popen(cmd, stdout=out_f, stderr=err_f)
                _, status, ru = os.wait4(p.pid, 0)
                wall_s = time.perf_counter() - t0
                assert status == 0
                out_f.seek(0)
                st = json.loads(out_f.read().decode())
                res_pair[b] = (wall_s, ru.ru_maxrss / 1024.0, st['fragments'])
            frags_pair[b] = parse_frags(os.path.join(outdir, 'chr21.frags'))
            shutil.rmtree(outdir, ignore_errors=True)
        ex_pct, rec_pct = compute_parity(frags_pair['cpu'], frags_pair['cuda'])
        ab_rows.append({
            'WL': wl, 'FL': fl,
            'cpu_wall_s': round(res_pair['cpu'][0], 2),
            'cuda_wall_s': round(res_pair['cuda'][0], 2),
            'speedup': round(res_pair['cpu'][0] / res_pair['cuda'][0], 2),
            'cpu_frags': res_pair['cpu'][2],
            'cuda_frags': res_pair['cuda'][2],
            'exact_%': round(ex_pct, 3),
            'recall_%': round(rec_pct, 4),
        })
    os.remove(hs21); os.remove(pt21)
    df_ab = pd.DataFrame(ab_rows)
    df_ab.to_csv('/content/ablation_wl_fl.csv', index=False)
    display(df_ab)